In [10]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=(
            "Real Madrid is one of the most successful football clubs in Europe. "
            "The club has won numerous UEFA Champions League titles and is known "
            "for its strong performances in European competitions."
        ),
        metadata={
            "source": "real_madrid.txt",
            "topic": "football_club",
            "club": "Real Madrid"
        }
    ),

    Document(
        page_content=(
            "Barcelona is one of the biggest football clubs in the world, but its "
            "European performances have been disappointing in recent seasons. "
            "Over roughly the last ten years, Barcelona has struggled to consistently "
            "compete at the highest level in the UEFA Champions League, with several "
            "early exits and disappointing knockout-stage results. Because of this, "
            "Barcelona can be described as an underperforming or unsuccessful club "
            "in European competition during this period, despite its strong domestic "
            "history and global reputation."
        ),
        metadata={
            "source": "barcelona.txt",
            "topic": "football_club",
            "club": "Barcelona"
        }
    ),

    Document(
        page_content=(
            "Bayern Munich is a major European football club with a strong record "
            "in the UEFA Champions League. The club regularly competes for major "
            "European trophies and has maintained a high level of performance "
            "against top European teams."
        ),
        metadata={
            "source": "bayern_munich.txt",
            "topic": "football_club",
            "club": "Bayern Munich"
        }
    ),

    Document(
        page_content=(
            "Manchester City has become one of the strongest clubs in European "
            "football in the modern era. The club has consistently competed deep "
            "into the UEFA Champions League and won its first Champions League "
            "title in 2023."
        ),
        metadata={
            "source": "manchester_city.txt",
            "topic": "football_club",
            "club": "Manchester City"
        }
    ),

    Document(
        page_content=(
            "Paris Saint-Germain is a prominent French football club that has "
            "invested heavily in building competitive squads. PSG has regularly "
            "participated in the UEFA Champions League and reached the final in 2020."
        ),
        metadata={
            "source": "psg.txt",
            "topic": "football_club",
            "club": "Paris Saint-Germain"
        }
    )
]

In [11]:
import os 
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

In [12]:
# Now you can access environment variables using os.getenv()
groq_api_key = os.getenv("GROQ_API_KEY")
huggingface_api_key = os.getenv("HUGGINGFACE_API_KEY")


In [13]:
# Initialize the ChatGroq model with the API key
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key
)


# Embedding model for generating vector representations of text
from langchain_huggingface import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# Vector store for storing and retrieving embeddings
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=hf_embeddings
)


# Similarity search
query = "What is LangChain?"

retrieved_docs = vector_store.similarity_search_with_score(
    query,
    k=2
)


# Display retrieved documents and scores
print("Retrieved Documents:\n")

for doc, score in retrieved_docs:
    print(f"Source: {doc.metadata['source']}")
    print(f"Topic: {doc.metadata['topic']}")
    print(f"Content: {doc.page_content}")
    print(f"Score: {score:.4f}")
    print("-" * 80)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3013.65it/s]


Retrieved Documents:

Source: psg.txt
Topic: football_club
Content: Paris Saint-Germain is a prominent French football club that has invested heavily in building competitive squads. PSG has regularly participated in the UEFA Champions League and reached the final in 2020.
Score: 1.9077
--------------------------------------------------------------------------------
Source: bayern_munich.txt
Topic: football_club
Content: Bayern Munich is a major European football club with a strong record in the UEFA Champions League. The club regularly competes for major European trophies and has maintained a high level of performance against top European teams.
Score: 1.9111
--------------------------------------------------------------------------------


In [14]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)
retriever.batch(["What is LangChain?", "Explain embeddings in AI."])

[[Document(id='645d6675-3334-4572-a28b-c792ff94450c', metadata={'source': 'psg.txt', 'topic': 'football_club', 'club': 'Paris Saint-Germain'}, page_content='Paris Saint-Germain is a prominent French football club that has invested heavily in building competitive squads. PSG has regularly participated in the UEFA Champions League and reached the final in 2020.'),
  Document(id='d07f5f10-8b43-4eb3-9b8f-47d71540fa7a', metadata={'source': 'bayern_munich.txt', 'topic': 'football_club', 'club': 'Bayern Munich'}, page_content='Bayern Munich is a major European football club with a strong record in the UEFA Champions League. The club regularly competes for major European trophies and has maintained a high level of performance against top European teams.')],
 [Document(id='4043f06f-09a4-4656-bc87-317d65b24158', metadata={'source': 'barcelona.txt', 'topic': 'football_club', 'club': 'Barcelona'}, page_content='Barcelona is one of the biggest football clubs in the world, but its European performan

In [16]:
# create a chain that uses the retriever to fetch relevant documents and then generates a response using the LLM

from langchain_core.prompts import  ChatPromptTemplate
from langchain_core.runnables import  RunnablePassthrough

message_template = """
You are a helpful assistant. Use the following context to answer the question.

{question}

context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("human", message_template)
])


rag_chain = {"context": retriever , "question": RunnablePassthrough()} | prompt | llm


queries = [
    "Which club has struggled in European competitions?",
    "What happened to Barcelona in European football?",
    "Which club won the Champions League in 2023?",
    "Which club reached the Champions League final in 2020?",
    "Which club is known for having many Champions League titles?"
]


# Run RAG


for query in queries:

    print(f"\nQuery: {query}")
    print("=" * 80)

    response = rag_chain.invoke(query)

    print("Answer:")
    print(response.content)


Query: Which club has struggled in European competitions?


Answer:
**Barcelona** has struggled in European competitions, with several early exits and disappointing knockout‑stage results over the last decade.

Query: What happened to Barcelona in European football?
Answer:
Barcelona’s recent European campaigns have been disappointing. Over the past decade the club has struggled to keep up with the top teams in the UEFA Champions League, suffering several early exits and underwhelming knockout‑stage performances. In short, while still a dominant force domestically and globally, Barcelona has been an under‑performing club in European competition during this period.

Query: Which club won the Champions League in 2023?
Answer:
Manchester City won the UEFA Champions League in 2023.

Query: Which club reached the Champions League final in 2020?
Answer:
The UEFA Champions League final in 2020 was contested by **Bayern Munich** and **Paris Saint‑Germain (PSG)**.

Query: Which club is known for having many Champions League titles?
Answer:
Real Madrid i